# Linearseparierbarer Datensatz mit Hebb-Regel und Perceptron

Dieses Notebook erzeugt einen künstlichen linearseparierbaren Datensatz, speichert ihn als Pandas DataFrame und trainiert anschließend ein Perceptron mit der Hebbschen Regel sowie mit dem klassischen Perceptron-Lernalgorithmus.

## 1. Bibliotheken und Datensatz

Der Datensatz besteht aus zwei gut getrennten Klassen. Die Punkte werden später als blaue und rote Punkte dargestellt.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

number_samples = 100

class_blue = np.random.normal(loc=[2,2], scale=0.5, size=(number_samples,2))
class_red = np.random.normal(loc=[5.5,6], scale=0.5, size=(number_samples,2))

features = np.vstack([class_blue, class_red])
labels = np.hstack([np.zeros(number_samples), np.ones(number_samples)])

data_frame = pd.DataFrame(features, columns=["x1","x2"])
data_frame["klasse"] = labels.astype(int)

data_frame.head()

## 2. Vorbereitung der Daten

Die Eingabedaten werden normiert und um einen Bias-Term erweitert.

In [ ]:
input_matrix = np.ones((len(labels),3))
input_matrix[:,0:2] = features

feature_min = input_matrix[:,0:2].min(axis=0)
feature_max = input_matrix[:,0:2].max(axis=0)

input_matrix[:,0:2] = (input_matrix[:,0:2]-feature_min)/(feature_max-feature_min)

def heaviside(value):
    return np.where(value > 0, 1, 0)

## 3. Lernen mit der Hebbschen Regel

In [ ]:
learning_rate = 0.25
hebb_weights = np.zeros(3)
hebb_targets = np.where(labels == 0, -1, 1)

for sample_index in range(len(hebb_targets)):
    hebb_weights += learning_rate * hebb_targets[sample_index] * input_matrix[sample_index]

print("Gewichte der Hebb-Regel:")
print(hebb_weights)

## 4. Lernen mit dem Perceptron

Die Implementierung orientiert sich an der vorgegebenen Vorlage.

In [ ]:
# Wir setzen den seed nochmal neu
np.random.seed(100)

learning_rate = 0.25
iteration = 0
maximum_iterations = 100000
weight_update = np.zeros(3)
weights = np.random.rand(3) - 0.5
convergence = 1

while convergence > 0 and iteration < maximum_iterations:
    iteration += 1

    sample_index = np.random.randint(len(labels))
    sample_input = input_matrix[sample_index,:]
    sample_label = labels[sample_index]

    error = sample_label - heaviside(weights @ sample_input)

    for feature_index in range(len(sample_input)):
        weight_update[feature_index] = learning_rate * error * sample_input[feature_index]
        weights[feature_index] += weight_update[feature_index]

    convergence = np.linalg.norm(labels - heaviside(weights @ input_matrix.T))

print("Anzahl der Iterationen:", iteration)
print("Gewichte des Perceptrons:")
print(weights)

## 5. Darstellung der Klassen und der Trenngeraden

Die Achsen werden fest auf x = 0 bis 7 und y = 0 bis 8 gesetzt. Zusätzlich wird die Grafik als SVG-Datei exportiert.

In [ ]:
colors = np.where(labels == 0, "blue", "red")

plt.figure(figsize=(8,6))
plt.scatter(features[:,0], features[:,1], c=colors)

line_x = np.linspace(0,7,300)
line_x_norm = (line_x-feature_min[0])/(feature_max[0]-feature_min[0])
line_y_norm = -(weights[2] + weights[0]*line_x_norm) / weights[1]
line_y = line_y_norm * (feature_max[1]-feature_min[1]) + feature_min[1]

plt.plot(line_x, line_y, color="black", linewidth=2, label="Trenngerade")
plt.xlim(0,7)
plt.ylim(0,8)
plt.xlabel("Merkmal x1")
plt.ylabel("Merkmal x2")
plt.grid(True)
plt.legend()
plt.show()